In [28]:
!pip install -q gradio groq langchain langchain-community langchain-groq langchain-huggingface langchain-text-splitters faiss-cpu sentence-transformers pypdf

In [29]:
import os
from google.colab import userdata

os.environ["GROQ_API_KEY"] = userdata.get('GROQ_API_KEY')
print("key is set")

key is set


In [8]:
from groq import Groq

client = Groq(api_key=os.environ["GROQ_API_KEY"])

response = client.chat.completions.create(
    model="openai/gpt-oss-120b",
    messages=[
        {"role": "system", "content": "You are a helpful assistant."},
        {"role": "user", "content": "Who won the world series in 2020?"},

    ]
)

print(response.choices[0].message.content)






The 2020 World Series was won by the **Los Angeles Dodgers**. They defeated the **Tampa Bay Rays** in six games (4–2) to claim the championship. It was the Dodgers’ seventh World Series title and their first since 1988.


In [9]:
from groq import Groq

client = Groq(api_key=os.environ["GROQ_API_KEY"])

response = client.chat.completions.create(
    model="llama-3.1-8b-instant",
    messages=[
        {"role": "system", "content": "You are an cricket coach and you teach cricket to students."},
        {"role": "user", "content": "Explain what is cricket, in two sentences."},
    ],
    )

print(response.choices[0].message.content)

Cricket is a popular team sport played with a bat and ball on a rectangular field with two sets of three stumps and two bails on each set, where one team sends two batsmen on the field to score runs while the other team tries to get them out. The objective of cricket is for the batting team to score runs by hitting the ball with the bat, while the bowling team tries to get the batsmen out by running them out, catching them, or dismissing them through different fielding restrictions.


In [10]:
import gradio as gr

SYSTEM_PROMPT = '''You are a football expert with experience in every aspect
of footbal rules and regulations including the history of football'''

def respond(message, history):
  messages = [{"role": "system", "content": SYSTEM_PROMPT}]
  for turn in history:
        messages.append({"role": turn["role"], "content": turn["content"]})
  messages.append({"role": "user", "content": message})
  response = client.chat.completions.create(
    model="llama-3.1-8b-instant",
    messages=messages,
    temperature=0.5
    )
  return response.choices[0].message.content

demo = gr.ChatInterface(fn=respond, type="messages", title="Football Expert")
demo.launch(debug=True)

It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://3159d9044a5d028f02.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


Keyboard interruption in main thread... closing server.
Killing tunnel 127.0.0.1:7860 <> https://22162f8d2f8af8eefd.gradio.live
Killing tunnel 127.0.0.1:7861 <> https://e430523e1ce634616d.gradio.live
Killing tunnel 127.0.0.1:7862 <> https://f3e3fc09e9f9e29cf7.gradio.live
Killing tunnel 127.0.0.1:7863 <> https://f1a0ea8c43e2cefc66.gradio.live
Killing tunnel 127.0.0.1:7864 <> https://3159d9044a5d028f02.gradio.live


In [11]:
import gradio as gr

SYSTEM_PROMPT = '''You are a football expert with experience in every aspect
of footbal rules and regulations including the history of football'''

def respond(message, history):
  messages = [{"role": "system", "content": SYSTEM_PROMPT}]
  for turn in history:
        messages.append({"role": turn["role"], "content": turn["content"]})
  messages.append({"role": "user", "content": message})
  stream = client.chat.completions.create(
    model="llama-3.1-8b-instant",
    messages=messages,
    temperature=0.5,
    stream=True,
  )


  partial = ""
  for chunk in stream:
    partial += chunk.choices[0].delta.content or ""
    yield partial

demo = gr.ChatInterface(fn=respond, type="messages", title="Football Expert")
demo.launch(debug=True)

It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://dd90ac1bffc8f2d354.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


Keyboard interruption in main thread... closing server.
Killing tunnel 127.0.0.1:7864 <> https://dd90ac1bffc8f2d354.gradio.live


In [ ]:
import gradio as gr
from groq import Groq
import os
from google.colab import userdata

# Initialize client
client = Groq(
    api_key=userdata.get('GROQ_API_KEY') # Correctly retrieve API key from Colab secrets
)

def respond(message, history, system_prompt, temperature):
    # Print received system prompt and temperature for debugging
    print(f"System Prompt received: {system_prompt}")
    print(f"Temperature received: {temperature}")

    # Build conversation
    messages = []

    # Add system prompt
    messages.append({"role": "system", "content": system_prompt})

    # Add previous chat history in the new 'messages' format
    for msg in history:
        messages.append({"role": msg["role"], "content": msg["content"]})

    # Add latest user message
    messages.append({"role": "user", "content": message})

    # Stream response
    response = client.chat.completions.create(
        model="llama-3.1-8b-instant",
        messages=messages,
        temperature=temperature,
        stream=True
    )

    partial_response = ""

    for chunk in response:
        if chunk.choices[0].delta.content:
            partial_response += chunk.choices[0].delta.content
            yield partial_response

demo = gr.ChatInterface(
    fn=respond,
    additional_inputs=[
        gr.Textbox(
            value="You are a helpful assistant.",
            label="System Prompt",
            lines=4
        ),
        gr.Slider(
            minimum=0,
            maximum=2,
            value=0.7,
            step=0.1,
            label="Temperature"
        )
    ],
    title="Customizable AI Chatbot",
    type="messages" # Ensure this is set for the new format

)

demo.launch(debug=True, share=True)

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://1138563c610967d038.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


System Prompt received: You are a helpful assistant.
Temperature received: 0.7
